# Study 2 · 01 · Ingest and QC

Loads `hf206-05` below-canopy PAR, applies the mandatory QC from `docs/00-research-program.md` §5, and audits above-canopy PAR continuity across HF004, HF282 and HF249.

**Gate G1:** ≥ 3 sensors survive QC with < 15 % loss, and at least one above-canopy source covers ≥ 70 % of growing-season daytime half-hours in ≥ 8 years.

Shared constants and QC functions live in `study2_lib.py`.

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

import study2_lib as s2

GROWING_SEASON = (120, 305)   # DOY, green-up through senescence
DAYLIGHT_HOURS = (6, 18)      # local clock hours where fAPAR is defined at all

pd.set_option("display.width", 140)
s2.selftest();

                         check                                                          got                                                         want  pass
         G(0) spherical == 0.5                                                      0.49967                                                          0.5  True
               K identity, x=1    [0.49967, 0.517297, 0.576969, 0.70664, 0.99934, 1.930577]    [0.49967, 0.517297, 0.576969, 0.70664, 0.99934, 1.930577]  True
               K identity, x=3 [0.828374, 0.831671, 0.843574, 0.873182, 0.956523, 1.322178] [0.828374, 0.831671, 0.843574, 0.873182, 0.956523, 1.322178]  True
multiplicative fAPAR(L=0) == 0                                                          0.0                                                          0.0  True
   printed fAPAR(L=0) == -0.06                                                        -0.06                                                        -0.06  True
          fAPAR(L=inf) -> Pinf                

## hf206-05 · below-canopy PAR

302,777 rows × 83 columns, 30-minute, 2008-01-01 → 2025-01-09. Six LI-190 SA quantum sensors at 1 m in named biometry plots, each with an `_ave`/`_max`/`_min` triplet.

The file is **not sorted by time** — row 0 is 2008-05-02. `load_hf206_ems` sorts on load.

QC applied, in order: `±7999` Campbell sentinels blanked across every numeric column; values outside physical range blanked; PAR below −10 blanked, PAR in [−10, 0) **clamped to 0**. The clamp is not cosmetic — small negatives are sensor zero-offset at low light, and discarding them would delete the high-SZA tail this study depends on.

In [2]:
raw = pd.read_csv(s2.PATHS["hf206_05"])
print(f"raw: {len(raw):,} rows × {raw.shape[1]} cols")

_, sentinels = s2.qc_sentinels(raw)
_, physical = s2.qc_physical(raw)

print(f"\n±7999 sentinels found in {len(sentinels)} columns:")
print(sentinels.to_string() if len(sentinels) else "  none")
print(f"\nout-of-physical-range values in {len(physical)} columns:")
print(physical.head(12).to_string() if len(physical) else "  none")

raw: 302,777 rows × 83 cols

±7999 sentinels found in 2 columns:
par_f2_max    60
swc3_max       6

out-of-physical-range values in 18 columns:
tair_us_w_20cm_max    66
tair_us_w_20cm_ave    62
tair_us_w_20cm_min    50
tair_us_e_20cm_max    48
tair_us_e_20cm_ave    46
tair_us_e_20cm_min    40
tair_us_e_1m_max      39
tair_us_e_1m_ave      33
tair_us_e_1m_min      18
swc3_max               9
tair_us_w_1m_min       6
swc4_max               3


In [3]:
par_cols = [f"{c}_{s}" for c in s2.SENSORS["col"] for s in ("ave", "max", "min")]
_, clamp = s2.clamp_par_floor(raw, par_cols)

clamp_tab = pd.DataFrame(
    [(c, v[0], v[1]) for c, v in clamp.items()],
    columns=["column", "blanked (< -10)", "clamped to 0 (-10..0)"],
)
clamp_tab[clamp_tab.iloc[:, 1:].sum(axis=1) > 0]

,column,blanked (< -10),clamped to 0 (-10..0)
0,par_c2_ave,0,4142
1,par_c2_max,0,520
2,par_c2_min,222,25210
3,par_e3_ave,0,7479
4,par_e3_max,0,2548
5,par_e3_min,62,23080
6,par_b2_ave,0,3640
7,par_b2_max,0,934
8,par_b2_min,114,43077
9,par_f3_ave,0,8837


In [4]:
ems = s2.load_hf206_ems()
par_ave = [f"{c}_ave" for c in s2.SENSORS["col"]]

n_rows = len(ems)
n_uniq = ems.datetime.nunique()
full = pd.date_range(ems.datetime.min(), ems.datetime.max(), freq="30min")

print(f"post-QC rows            : {n_rows:,}   {ems.datetime.min()} → {ems.datetime.max()}")
print(f"unique timestamps       : {n_uniq:,}")
print(f"surplus (duplicate) rows: {n_rows - n_uniq:,}")
print(f"expected 30-min slots   : {len(full):,}")
print(f"genuinely absent slots  : {len(full) - n_uniq:,}  ({(len(full)-n_uniq)/len(full)*100:.1f} %)")

# Exact copies, or genuinely different measurements at the same timestamp?
dup = ems[ems.datetime.duplicated(keep=False)]
identical = dup.groupby("datetime")[par_ave].apply(lambda x: len(x.drop_duplicates()) == 1)
print(f"\nduplicated timestamps   : {len(identical):,}")
print(f"  exact copies          : {int(identical.sum()):,}")
print(f"  genuinely differing   : {int((~identical).sum()):,}")
print("\nduplicate rows by year:")
print(dup.datetime.dt.year.value_counts().sort_index().to_string())

# dec_hour is an independent record of the time. Where it disagrees with the
# datetime clock, the two disagree about the offset -- a DST tell, and evidence
# for Phase 2's interval-labelling question.
clock = ems.datetime.dt.hour + ems.datetime.dt.minute / 60
print(f"\nrows where dec_hour != clock hour: {int((ems.dec_hour - clock).abs().gt(1e-6).sum()):,}")

post-QC rows            : 302,777   2008-01-01 00:00:00-05:00 → 2025-01-09 11:30:00-05:00
unique timestamps       : 292,601
surplus (duplicate) rows: 10,176
expected 30-min slots   : 298,488
genuinely absent slots  : 5,887  (2.0 %)

duplicated timestamps   : 10,175
  exact copies          : 10,174
  genuinely differing   : 1

duplicate rows by year:
datetime
2010        2
2024    20348

rows where dec_hour != clock hour: 4,260


### Timestamp integrity — an archive defect

**2024 is duplicated wholesale in the file.** 10,175 timestamps appear twice, of which 10,174 are byte-identical in their PAR values; 20,348 of the 20,350 affected rows fall in 2024. This is a file-assembly artifact, not repeated measurement. Exactly one timestamp (2010-05-27 10:00) carries genuinely differing values and needs a separate decision.

It does not touch this study — HF004 ends 2023-12-31, so 2024 is outside the analysis window either way — but it is a reportable archive error, alongside the ±7999 sentinels and the EML unit labels, for `hf-im@lists.fas.harvard.edu`.

Deduplicate on `datetime` before any resampling. Naive row counts are misleading: 302,777 rows cover 292,601 unique timestamps, and **5,887 30-minute slots (2.0 %) are genuinely absent** — the implicit missingness the catalog's row-level "essentially gap-free" claim obscures.

The `dec_hour` disagreement is a separate signal and feeds Phase 2: it is an independent record of the clock time, so where it disagrees with the parsed `datetime` the file is internally inconsistent about its own offset.

### Per-sensor missingness

Recomputed from the data rather than copied from the catalog. Agreement confirms QC is clamping rather than discarding — a pipeline that dropped the small negatives instead would show inflated loss here.

Logger attribution follows the HF206 **methods text**, not the `_ave` column descriptions, which wrongly assign all six sensors to EMS1.

In [5]:
CATALOG_MISSING = {"C2": 1.4, "E3": 1.4, "B2": 1.6, "F3": 7.6, "F2": 11.0, "C3": 12.1}


def sensor_missingness(df):
    """Post-QC missingness per sensor, against the catalog's published value."""
    out = s2.SENSORS.copy()
    out["missing_%"] = [df[f"{c}_ave"].isna().mean() * 100 for c in out["col"]]
    out["catalog_%"] = out["plot"].map(CATALOG_MISSING)
    out["delta"] = (out["missing_%"] - out["catalog_%"]).round(2)
    out["primary"] = out["plot"].isin(s2.PRIMARY_PLOTS)
    return out.round(2)


miss = sensor_missingness(ems)
miss

,plot,logger,col,wholly_inside,modis_cell,missing_%,catalog_%,delta,primary
0,C2,EMS1,par_c2,False,pixel_4,1.36,1.4,-0.04,True
1,E3,EMS2,par_e3,True,pixel_1,1.38,1.4,-0.02,True
2,B2,EMS1,par_b2,False,pixel_4,1.64,1.6,0.04,True
3,F3,EMS2,par_f3,False,pixel_1,7.61,7.6,0.01,False
4,F2,EMS2,par_f2,False,pixel_1,10.96,11.0,-0.04,False
5,C3,EMS1,par_c3,False,pixel_4,12.14,12.1,0.04,False


## Above-canopy PAR continuity — the Gate G1 question

Flagged "continuity not yet verified" in both `00-research-program.md` and `01-data-catalog.md`, and load-bearing: without a denominator there is no fAPAR.

Candidate columns, resolved by inspecting the real files rather than the catalog:

| Source | Column | Role |
|---|---|---|
| `hf004-01` | `par.29` | incident above canopy, EMS tower — **eq 3 `PAR_i`** |
| `hf004-01` | `par.ref.ue.m2.s` | reflected — **eq 3 `PAR_cr`** |
| `hf004-01` | `par.dfs` / `par.dir` | **measured** diffuse and direct PAR |
| `hf004-02` | `par.28m.filled...` | gap-filled incident |
| `hf282-01` | `parac_up` / `parac_down` | HDW tower, ~400 m SW |
| `hf249-01` | `diffuse` / `direct` | Barn tower, independent diffuse |

Coverage is measured over **growing-season daylight only** — night-time completeness is irrelevant and would flatter every source.

In [6]:
def coverage_by_year(path, time_col, value_col, label):
    """Non-null fraction per year over growing-season daylight hours."""
    a = pd.read_csv(path, usecols=[time_col, value_col])
    a[time_col] = pd.to_datetime(a[time_col])
    a = a[a[time_col].dt.dayofyear.between(*GROWING_SEASON)
          & a[time_col].dt.hour.between(*DAYLIGHT_HOURS)]
    cov = a.groupby(a[time_col].dt.year)[value_col].apply(lambda s: s.notna().mean() * 100)
    return cov.rename(label)


SOURCES = [
    (s2.PATHS["hf004_01"], "datetime", "par.29",                     "hf004 par_i"),
    (s2.PATHS["hf004_02"], "datetime", "par.28m.filled.e.6mol.m2.s", "hf004 filled"),
    (s2.PATHS["hf004_01"], "datetime", "par.ref.ue.m2.s",            "hf004 par_cr"),
    (s2.PATHS["hf004_01"], "datetime", "par.dfs.ue.m2.s",            "hf004 diffuse"),
    (s2.PATHS["hf282_01"], "datetime", "parac_up",                   "hf282 par_i"),
    (s2.PATHS["hf282_01"], "datetime", "parac_down",                 "hf282 par_cr"),
    (s2.PATHS["hf249_01"], "datetime", "diffuse",                    "hf249 diffuse"),
]

cov = pd.concat([coverage_by_year(*s) for s in SOURCES], axis=1)
cov = cov.loc[cov.index >= 2008].round(0)
cov

,hf004 par_i,hf004 filled,hf004 par_cr,hf004 diffuse,hf282 par_i,hf282 par_cr,hf249 diffuse
datetime,,,,,,,
2008,100.0,100.0,0.0,0.0,NaN,NaN,NaN
2009,100.0,100.0,0.0,0.0,NaN,NaN,NaN
2010,100.0,100.0,0.0,0.0,NaN,NaN,NaN
2011,100.0,100.0,0.0,0.0,NaN,NaN,NaN
2012,100.0,100.0,0.0,92.0,NaN,NaN,100.0
2013,100.0,100.0,0.0,100.0,NaN,NaN,100.0
2014,100.0,100.0,0.0,79.0,100.0,100.0,100.0
2015,100.0,100.0,0.0,100.0,100.0,100.0,100.0
2016,100.0,100.0,0.0,99.0,100.0,100.0,80.0


In [7]:
fig = px.imshow(
    cov.T, text_auto=".0f", aspect="auto", origin="upper",
    color_continuous_scale=[(0.0, "#7d2b2b"), (0.7, "#c9a227"), (1.0, "#2d6a4f")],
    range_color=[0, 100],
    labels=dict(x="year", y="", color="coverage %"),
)
fig.update_layout(
    title="Above-canopy PAR coverage · growing-season daylight (DOY 120–305, 06–18 h)",
    template="plotly_white", height=380, margin=dict(l=110, r=20, t=60, b=40),
    coloraxis_colorbar=dict(title="%"),
)
fig.update_xaxes(dtick=1, tickangle=0)
fig.show()

### Reading the heatmap

`par.29` is at 100 % every year 2008–2023 — the continuity risk is closed, and the gap-filled column is not needed.

Two real constraints, both new:

- **`par.ref` (`PAR_cr`, eq 3) begins only in 2018.** Zero coverage 2008–2017. `hf282 par_cr` (`parac_down`, 2014–) is the fallback but sits at a different tower ~400 m away.
- **HF004 ends 2023-12-31** while `hf206-05` runs to 2025-01-09, so the joint window is **2008–2023**, not 2008–2025.

Measured diffuse PAR from 2012 means diffuse fraction is *observed* rather than derived from a clearness index for most of the record — HF249 corroborates independently for 2012–2017.

## How much does dropping `PAR_cr` actually matter?

Eq 3 is `Φ = PAR_i − PAR_cr − PAR_t`. If canopy-reflected PAR is a small and stable fraction of incident, the 2008–2017 gap can be closed with a constant rather than losing ten years. Estimated on 2018–2023, where both columns exist.

In [8]:
hf4 = pd.read_csv(s2.PATHS["hf004_01"],
                  usecols=["datetime", "par.29", "par.ref.ue.m2.s"])
hf4["datetime"] = pd.to_datetime(hf4["datetime"])
both = hf4[hf4["par.29"].gt(s2.PAR_MIN) & hf4["par.ref.ue.m2.s"].notna()
           & hf4["datetime"].dt.dayofyear.between(*GROWING_SEASON)].copy()
both["albedo_par"] = both["par.ref.ue.m2.s"] / both["par.29"]

print(f"n = {len(both):,}   {both.datetime.dt.year.min()}–{both.datetime.dt.year.max()}")
print(both["albedo_par"].describe(percentiles=[.05, .25, .5, .75, .95]).round(4).to_string())
print("\nPAR albedo by year:")
print(both.groupby(both.datetime.dt.year)["albedo_par"].median().round(4).to_string())

n = 10,314   2018–2023
count    10314.0000
mean         0.0493
std          0.0125
min          0.0000
5%           0.0345
25%          0.0408
50%          0.0470
75%          0.0563
95%          0.0715
max          0.2669

PAR albedo by year:
datetime
2018    0.0530
2019    0.0559
2020    0.0433
2021    0.0440
2022    0.0441
2023    0.0393


In [9]:
med = both["albedo_par"].median()
iqr = both["albedo_par"].quantile([.25, .75]).to_numpy()

fig = go.Figure()
fig.add_histogram(x=both["albedo_par"].clip(0, 0.15), nbinsx=90,
                  marker_color="#2d6a4f", opacity=0.85, name="observed")
fig.add_vline(x=med, line_color="#7d2b2b", line_width=2,
              annotation_text=f"median {med:.3f}", annotation_position="top right")
fig.update_layout(
    title="Canopy PAR albedo (PAR_cr / PAR_i), growing season 2018–2023",
    xaxis_title="PAR_cr / PAR_i", yaxis_title="count",
    template="plotly_white", height=360, margin=dict(l=60, r=20, t=60, b=50),
    showlegend=False,
)
fig.show()

print(f"median {med:.4f}   IQR {iqr[0]:.4f}–{iqr[1]:.4f}")
print(f"absolute fAPAR error if PAR_cr is simply omitted: ≈ {med:.3f}")
print(f"error if replaced by the constant median instead: ≈ {(iqr[1]-iqr[0])/2:.4f}")

median 0.0470   IQR 0.0408–0.0563
absolute fAPAR error if PAR_cr is simply omitted: ≈ 0.047
error if replaced by the constant median instead: ≈ 0.0077


## Gate G1

In [10]:
sensors_ok = int((miss["missing_%"] < 15).sum())
above_years = int((cov["hf004 par_i"] >= 70).sum())
catalog_ok = bool((miss["delta"].abs() < 0.2).all())

checks = pd.DataFrame([
    ("sensors < 15 % loss",              sensors_ok,  "≥ 3",  sensors_ok >= 3),
    ("years above-canopy ≥ 70 %",        above_years, "≥ 8",  above_years >= 8),
    ("missingness matches catalog",      catalog_ok,  "True", catalog_ok),
], columns=["check", "value", "required", "pass"])

print(checks.to_string(index=False))
print("\nGATE G1:", "PASS" if checks["pass"].all() else "FAIL")

                      check value required  pass
        sensors < 15 % loss     6      ≥ 3  True
  years above-canopy ≥ 70 %    16      ≥ 8  True
missingness matches catalog  True     True  True

GATE G1: PASS


## Carried forward

- Analysis window is **2008–2023** (HF004 ends 2023-12-31).
- `PAR_i` = `hf004-01` `par.29`, raw not gap-filled — it needs no filling.
- `PAR_cr` handling is decided by the albedo result above: measured 2018–2023, constant elsewhere, with the substitution flagged as a covariate so its effect on the SZA response can be tested directly.
- Diffuse fraction is **measured** from 2012 (HF004, corroborated by HF249); only 2008–2011 needs an Erbs/clearness-index derivation.
- Next: `Study2_02_SolarGeometry` — the interval-labelling half of the timestamp risk is still open. Timezone is settled as EST year-round.